# Plot ARSO in-situ data (Slovenian lakes)

Plots the Level2 products for every station in `notes/stations.json`:
- depth-time heatmaps of probe temperature (`Temp`) and probe chlorophyll (`Chla_probe`)
- time series of Secchi depth, lab chlorophyll (`Chla_lab`) and nutrients for each sample layer (upper, column, mid, deep)

Values with `_qual > 0` are already masked (NaN) in Level2. Run `python scripts/main.py` first.

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

DATA = "../data/Level2"
with open("../notes/stations.json", encoding="utf-8") as f:
    stations = json.load(f)


def load(key, product):
    """Concatenate the yearly Level2 files of one station along time."""
    files = sorted(glob.glob(os.path.join(DATA, key, "L2_{}_{}_*.nc".format(product, key))))
    if not files:
        return None
    return xr.concat([xr.open_dataset(f) for f in files], dim="time", data_vars="all").sortby("time")

## Profiles: depth-time heatmaps

In [ ]:
# Each yearly file has its own depth axis (only the depths sampled that year), so each file is drawn
# separately on the same axes - the way Datalakes combines them.
for key, s in stations.items():
    files = sorted(glob.glob(os.path.join(DATA, key, "L2_Profiles_{}_*.nc".format(key))))
    if not files:
        continue
    datasets = [xr.open_dataset(f) for f in files]
    variables = [v for v in ["Temp", "Chla_probe", "Turbidity"] if v in datasets[0]]
    fig, axes = plt.subplots(len(variables), 1, figsize=(16, 3.5 * len(variables)), squeeze=False)
    fig.suptitle("{} - {}".format(s["lake_long_name"], s["station_name"]))
    for ax, var in zip(axes[:, 0], variables):
        vmin = np.nanmin([np.nanmin(ds[var].values) if np.isfinite(ds[var].values).any() else np.nan for ds in datasets])
        vmax = np.nanmax([np.nanmax(ds[var].values) if np.isfinite(ds[var].values).any() else np.nan for ds in datasets])
        cmap = "RdYlBu_r" if var == "Temp" else "viridis"
        m = None
        for ds in datasets:
            if not np.isfinite(ds[var].values).any():
                continue
            if ds.sizes["depth"] > 1:
                m = ax.pcolormesh(ds.time.values, ds.depth.values, ds[var].values, shading="nearest", cmap=cmap, vmin=vmin, vmax=vmax)
            else:
                m = ax.scatter(ds.time.values, np.repeat(ds.depth.values, ds.sizes["time"]), c=ds[var].values[0], cmap=cmap, vmin=vmin, vmax=vmax, s=12)
        ax.set_ylim(max(float(ds.depth.max()) for ds in datasets) + 0.5, 0)
        ax.set_ylabel("Depth (m)")
        ax.set_title(var)
        if m is not None:
            fig.colorbar(m, ax=ax, label="{} ({})".format(var, datasets[0][var].attrs.get("units", "")))
    plt.tight_layout()
    plt.show()

## Samples: Secchi depth, lab chlorophyll and nutrients

One panel per variable; each sample layer is a separate line. The sampled depth range of each value is in `<var>_depth_top` / `<var>_depth_bottom`.

In [ ]:
LAYERS = {"": "upper", "_column": "column", "_mid": "mid", "_deep": "deep"}
for key, s in stations.items():
    ds = load(key, "Samples")
    if ds is None:
        continue
    variables = [v for v in ["Secchi", "Chla_lab", "TN", "TP", "NH4", "NO3", "NO2"] if v in ds]
    fig, axes = plt.subplots(len(variables), 1, figsize=(16, 2.6 * len(variables)), sharex=True, squeeze=False)
    fig.suptitle("{} - {}".format(s["lake_long_name"], s["station_name"]))
    for ax, var in zip(axes[:, 0], variables):
        for suffix, label in LAYERS.items():
            name = var + suffix
            if name in ds:
                ax.plot(ds.time.values, ds[name].values, "o-", ms=3, lw=0.8, label=label)
        ax.set_ylabel("{} ({})".format(var, ds[var].attrs.get("units", "")))
        if var == "Secchi":
            ax.invert_yaxis()
        ax.legend(loc="upper left", fontsize=8, ncol=4)
    plt.tight_layout()
    plt.show()